# 🧠 Predicting Mental Health Diagnosis at Workplace

> **Goal:** Predict whether an employee has a mental health diagnosis based on their work environment, habits, and company support — using real ML models that score **~97% accuracy**.

---

## 📋 Table of Contents
1. [Import Libraries](#1)
2. [Load & Explore Data](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Data Preprocessing](#4)
5. [Train ML Models](#5)
6. [Evaluate & Compare Models](#6)
7. [Feature Importance](#7)
8. [Conclusion](#8)

---

### 📌 About this Dataset
- **10,000 employee records** from various countries and industries
- **34 features** covering work habits, stress, burnout, and company support
- **Target → `has_diagnosis`** : Does the employee have a mental health diagnosis? (`Yes` / `No`)

### 🚫 Why NOT use `intention_to_leave` as the target?
The `intention_to_leave` column has **5 classes** with no real signal —  
all models scored only ~25% (close to random guessing at 20%). This happens in synthetic datasets  
where the target is generated independently of the features.  
We switched to `has_diagnosis`, which has **genuine signal** and meaningful real-world use.

## 1. Import Libraries <a id='1'></a>

In [ ]:
# ── Data ──────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualization ─────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Preprocessing ─────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# ── Models ────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# ── Metrics ───────────────────────────────
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
SEED = 42

print('✅ All libraries imported!')

## 2. Load & Explore Data <a id='2'></a>

In [ ]:
df = pd.read_csv('/kaggle/input/mental-health-workplace/mental_health_workplace.csv')

print(f'📐 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Quick info — column types and non-null counts
df.info()

In [ ]:
# Statistical summary
df.describe().round(2)

In [ ]:
# Missing values check
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
mv = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
mv = mv[mv['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print(f'⚠️  {len(mv)} columns have missing values')
mv

In [ ]:
# Our TARGET variable
print('🎯 Target: has_diagnosis')
vc = df['has_diagnosis'].value_counts()
print(vc)
print(f'\nBalance → No: {vc["No"]/len(df)*100:.1f}%  |  Yes: {vc["Yes"]/len(df)*100:.1f}%')

## 3. Exploratory Data Analysis (EDA) <a id='3'></a>

Let's **visualize patterns** in the data before building models.

In [ ]:
# ── Plot 1: Target distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#66C2A5', '#FC8D62']

# Bar chart
vc = df['has_diagnosis'].value_counts()
bars = axes[0].bar(vc.index, vc.values, color=colors, width=0.5, edgecolor='white')
axes[0].set_title('🎯 Mental Health Diagnosis Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Employees')
for b in bars:
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 50,
                 f'{int(b.get_height()):,}', ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(vc.values, labels=vc.index, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('📊 Proportion of Diagnoses', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable Overview', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Key numeric features by diagnosis ─────────────────────────────────
num_features_to_plot = [
    'absenteeism_days_per_year', 'burnout_risk_score',
    'productivity_score', 'job_satisfaction_score'
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, feat in zip(axes, num_features_to_plot):
    sns.boxplot(data=df, x='has_diagnosis', y=feat, palette='Set2', ax=ax)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.set_xlabel('Has Diagnosis')

plt.suptitle('📦 Key Features: Diagnosed vs Not Diagnosed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Stress level breakdown ───────────────────────────────────────────
ct = pd.crosstab(df['stress_level'], df['has_diagnosis'], normalize='index') * 100

ct.plot(kind='bar', figsize=(8, 4), color=['#66C2A5', '#FC8D62'],
        edgecolor='white', width=0.6)

plt.title('😓 Stress Level vs Mental Health Diagnosis (%)', fontsize=13, fontweight='bold')
plt.xlabel('Stress Level')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Has Diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 4: Employer support & EAP availability ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col in zip(axes, ['employer_support_level', 'eap_available']):
    ct2 = pd.crosstab(df[col], df['has_diagnosis'], normalize='index') * 100
    ct2.plot(kind='bar', ax=ax, color=['#66C2A5', '#FC8D62'],
             edgecolor='white', width=0.6)
    ax.set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Percentage (%)')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='Has Diagnosis')

plt.suptitle('🏢 Employer Support vs Diagnosis Rates', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 5: Correlation heatmap (numeric columns) ──────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c not in ['year']]

plt.figure(figsize=(12, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            annot_kws={'size': 8})
plt.title('🔗 Correlation Between Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing <a id='4'></a>

We need to:
1. **Remove irrelevant or directly leaky columns** (`mental_health_condition`, `treatment_type` directly reveal the diagnosis)
2. **Handle missing values**
3. **Encode categorical columns** (text → numbers)
4. **Scale numeric columns** so all values are on the same range

In [ ]:
# ── Step 1: Remove leaky / ID columns ────────────────────────────────────────
# mental_health_condition  → directly tells the condition (leaks the target)
# treatment_type           → only exists if there IS a diagnosis (leaks the target)
# record_id, year          → identifiers, not features
# intention_to_leave       → we are NOT predicting this

LEAKY_COLS = ['record_id', 'year', 'intention_to_leave',
              'mental_health_condition', 'treatment_type']

df_model = df.drop(columns=LEAKY_COLS)

# Separate features and target
X = df_model.drop(columns=['has_diagnosis'])
y_raw = df_model['has_diagnosis'].fillna('No')  # fill the few missing ones

# Encode target: No → 0, Yes → 1
le_target = LabelEncoder()
y = le_target.fit_transform(y_raw)

print(f'✅ Features (X) : {X.shape[1]} columns, {X.shape[0]:,} rows')
print(f'✅ Target  (y) : {dict(zip(le_target.classes_, np.bincount(y)))}')

In [ ]:
# ── Step 2: Identify numeric vs categorical columns ───────────────────────────
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

print(f'📊 Numeric columns    ({len(num_cols)}): {num_cols}')
print(f'\n📝 Categorical columns ({len(cat_cols)}): {cat_cols}')

In [ ]:
# ── Step 3: Build preprocessing pipeline ─────────────────────────────────────
# A Pipeline chains multiple steps, so preprocessing is clean and reproducible.

# For numeric: fill missing with median, then scale (mean=0, std=1)
num_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler())
])

# For categorical: fill missing with most common value, then one-hot encode
cat_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('encode',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

print('✅ Preprocessing pipeline created!')

In [ ]:
# ── Step 4: Train / Test split ────────────────────────────────────────────────
# 80% for training the model, 20% for final evaluation
# stratify=y → ensures both splits have same class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Apply preprocessing
X_train_p = preprocessor.fit_transform(X_train)  # fit on train data only!
X_test_p  = preprocessor.transform(X_test)        # transform test with same params

print(f'🏋️  Training set : {X_train_p.shape[0]:,} samples, {X_train_p.shape[1]} features')
print(f'🧪 Testing  set : {X_test_p.shape[0]:,} samples, {X_test_p.shape[1]} features')

## 5. Train ML Models <a id='5'></a>

We train **5 models** from simple to complex. This lets beginners see how performance improves with more powerful algorithms.

In [ ]:
# Define all models
models = {
    '📐 Logistic Regression'  : LogisticRegression(max_iter=5000, solver='saga', random_state=SEED),
    '🌳 Decision Tree'        : DecisionTreeClassifier(max_depth=8, random_state=SEED),
    '🌲 Random Forest'        : RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    '📈 Gradient Boosting'    : GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                           max_depth=5, random_state=SEED),
    '⚡ XGBoost'              : XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                                              subsample=0.8, colsample_bytree=0.8,
                                              random_state=SEED, eval_metric='logloss', n_jobs=-1)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = {}

print('🚀 Training all models...\n')
print(f'{"Model":<28} {"Test Acc":>10} {"ROC-AUC":>10} {"CV Mean":>10} {"CV Std":>8}')
print('─' * 72)

for name, model in models.items():
    # Train
    model.fit(X_train_p, y_train)

    # Test set predictions
    y_pred      = model.predict(X_test_p)
    y_pred_prob = model.predict_proba(X_test_p)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_prob)

    # 5-Fold Cross Validation (on training data)
    cv_scores = cross_val_score(model, X_train_p, y_train, cv=cv, scoring='accuracy')

    results[name] = {
        'model'     : model,
        'y_pred'    : y_pred,
        'y_prob'    : y_pred_prob,
        'accuracy'  : acc,
        'roc_auc'   : auc,
        'cv_mean'   : cv_scores.mean(),
        'cv_std'    : cv_scores.std()
    }

    print(f'{name:<28} {acc:>10.4f} {auc:>10.4f} {cv_scores.mean():>10.4f} {cv_scores.std():>8.4f}')

print('\n✅ All models trained!')

## 6. Evaluate & Compare Models <a id='6'></a>

In [ ]:
# ── Comparison DataFrame ──────────────────────────────────────────────────────
comp_df = pd.DataFrame({
    'Model'      : list(results.keys()),
    'Test Acc'   : [v['accuracy'] for v in results.values()],
    'ROC-AUC'    : [v['roc_auc']  for v in results.values()],
    'CV Mean'    : [v['cv_mean']  for v in results.values()],
    'CV Std'     : [v['cv_std']   for v in results.values()]
}).sort_values('Test Acc', ascending=False).reset_index(drop=True)

print('📊 Model Comparison:')
comp_df.style.background_gradient(subset=['Test Acc','ROC-AUC'], cmap='Greens')

In [ ]:
# ── Dual bar chart: Accuracy vs ROC-AUC ──────────────────────────────────────
model_names = [n.split()[-1] for n in results.keys()]   # short names for axis
accs = [v['accuracy'] for v in results.values()]
aucs = [v['roc_auc']  for v in results.values()]

x = np.arange(len(model_names))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, accs, w, label='Test Accuracy', color='#66C2A5', edgecolor='white')
b2 = ax.bar(x + w/2, aucs, w, label='ROC-AUC',       color='#FC8D62', edgecolor='white')

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() - 0.04,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold', color='white')

ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylim(0.88, 1.01)
ax.set_ylabel('Score')
ax.set_title('🏆 Model Performance: Accuracy vs ROC-AUC', fontsize=13, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.tight_layout()
plt.show()

In [ ]:
# ── Pick Best Model ───────────────────────────────────────────────────────────
best_name = max(results, key=lambda k: results[k]['accuracy'])
best = results[best_name]

print(f'🥇 Best Model   : {best_name}')
print(f'   Test Accuracy: {best["accuracy"]:.4f} ({best["accuracy"]*100:.2f}%)')
print(f'   ROC-AUC      : {best["roc_auc"]:.4f}\n')

print('📋 Detailed Classification Report:')
print(classification_report(y_test, best['y_pred'],
                             target_names=le_target.classes_))

In [ ]:
# ── Confusion Matrix + ROC Curve (side by side) ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, best['y_pred'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_target.classes_)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'🔷 Confusion Matrix — {best_name.split()[-1]}',
                  fontsize=12, fontweight='bold')

# ROC Curve for all models
for name, res in results.items():
    short = name.split()[-1]
    RocCurveDisplay.from_predictions(
        y_test, res['y_prob'],
        name=f"{short} (AUC={res['roc_auc']:.3f})",
        ax=axes[1]
    )
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_title('📈 ROC Curve — All Models', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()

## 7. Feature Importance <a id='7'></a>

**Which factors matter most** in predicting a mental health diagnosis?

In [ ]:
# Get feature names after one-hot encoding
ohe_names = (preprocessor
             .named_transformers_['cat']['encode']
             .get_feature_names_out(cat_cols).tolist())
all_feat_names = num_cols + ohe_names

# Use the best model's feature importances
importances = best['model'].feature_importances_
feat_imp = (pd.DataFrame({'Feature': all_feat_names, 'Importance': importances})
              .sort_values('Importance', ascending=False)
              .head(20))

# Clean up feature names for readability
feat_imp['Feature'] = feat_imp['Feature'].str.replace('_', ' ').str.title()

plt.figure(figsize=(10, 7))
palette = sns.color_palette('viridis', len(feat_imp))
bars = plt.barh(feat_imp['Feature'][::-1],
                feat_imp['Importance'][::-1], color=palette)
plt.xlabel('Feature Importance Score')
plt.title('🔑 Top 20 Most Important Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔝 Top 10 Features:')
print(feat_imp.head(10).to_string(index=False))

In [ ]:
# ── Distribution of #1 feature by target ─────────────────────────────────────
top_feature = feat_imp.iloc[0]['Feature'].lower().replace(' ', '_')

plt.figure(figsize=(9, 4))
for label, color in zip(['No', 'Yes'], ['#66C2A5', '#FC8D62']):
    subset = df[df['has_diagnosis'] == label][top_feature].dropna()
    subset.hist(bins=30, alpha=0.7, label=f'Diagnosis: {label}', color=color)

plt.xlabel(top_feature.replace('_', ' ').title())
plt.ylabel('Count')
plt.title(f'📊 Distribution of "{top_feature.replace("_", " ").title()}" by Diagnosis',
          fontsize=12, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Conclusion <a id='8'></a>

---

### ✅ What We Built
| Step | Description |
|------|-------------|
| EDA | Explored 10,000 employee records with 5 charts revealing key patterns |
| Target | `has_diagnosis` — binary (Yes/No) — meaningful, real signal |
| Preprocessing | Imputation + StandardScaler + OneHotEncoding via sklearn Pipeline |
| Models | 5 classifiers from simple (Logistic Regression) to advanced (XGBoost) |
| Evaluation | Test Accuracy, ROC-AUC, 5-Fold CV, Confusion Matrix, ROC Curve |

---

### 🏆 Final Results
| Model | Test Accuracy | ROC-AUC |
|---|---|---|
| ⚡ XGBoost | **~96.8%** | **~0.996** |
| 📐 Logistic Regression | ~96.5% | ~0.994 |
| 📈 Gradient Boosting | ~96.0% | ~0.994 |
| 🌲 Random Forest | ~95.7% | ~0.991 |
| 🌳 Decision Tree | ~94.9% | ~0.967 |

---

### 💡 Key Insights
1. **Absenteeism days** is the strongest predictor — employees with diagnoses take significantly more days off
2. **Burnout risk score** and **productivity score** are also strong signals
3. **Employer support level** and **EAP availability** show meaningful differences in diagnosis rates
4. **High stress** employees are more likely to have a mental health diagnosis

---

### 🚀 What You Can Try Next
- **Hyperparameter tuning** with `Optuna` for even better XGBoost performance
- **SHAP values** for model explainability
- **Threshold tuning** to balance precision vs recall based on business need
- **Cross-industry analysis** — does model performance differ by industry?

---

> 💬 **Found this notebook useful? Give it an upvote ⬆️ and drop a comment below!**